# WMMD Dataset Download & Exploration

**Project:** Spectral Graph Methods for Marine Mammal Vocalization Classification  
**Course:** MATH 5466 — Mathematics of Machine Learning and Data Analysis II

This notebook downloads the Watkins Marine Mammal Sound Database from HuggingFace and performs initial exploration to inform data decisions for all three project levels:

- **Level 1** — broad taxonomic group separation (baleen whales / dolphins / toothed whales / pinnipeds)
- **Level 2** — within-Delphinidae species classification under low-label conditions
- **Level 3** — within-species vocalization-type discovery (Bottlenose Dolphin, fallback species if too few clips)

Audio is decoded manually with `soundfile` to avoid the `torchcodec` DLL issue on Windows.

## 0. Dependencies

Run once. `datasets<4.0` keeps audio decoding on `soundfile` and avoids the torchcodec mess; if you already have `datasets>=4.0`, the `decode=False` cast below sidesteps it anyway, so this pin is optional.

In [ ]:
# Install the repository's requirements.txt into this notebook kernel first.


In [ ]:
import os
import io
import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
from collections import Counter

from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "utils" / "graph.py").is_file() and
           (PROJECT_ROOT / "v2" / "utils" / "splits.py").is_file()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Start Jupyter inside the cloned repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "figures"
os.makedirs(DATA_DIR, exist_ok=True)
print(f"Data directory: {DATA_DIR}")

## 1. Download the Dataset

The WMMD is hosted on HuggingFace as `confit/wmms-parquet`. It contains 1,697 audio clips across 32 species, pre-split 80/20 (1357 train, 340 test). Total size ~1.22 GB. Cached locally on first run.

We cast the `audio` column with `decode=False` so HuggingFace hands us raw WAV bytes rather than trying to decode through `torchcodec`. We then decode each clip on demand with `soundfile`.

In [ ]:
from datasets import load_dataset, load_from_disk, Audio

# Revision recorded in the original local cache; preserve row order for clip IDs.
DATASET_ID = "confit/wmms-parquet"
DATASET_REVISION = "a90a38e0006991f0c6f6d4e05261949a1da7f14e"
HF_LOCAL = PROJECT_ROOT / "Mss" / "data" / "wmms_parquet_local"
if HF_LOCAL.exists():
    ds = load_from_disk(str(HF_LOCAL))
else:
    ds = load_dataset(DATASET_ID, revision=DATASET_REVISION)
    ds = ds.cast_column("audio", Audio(decode=False))
    HF_LOCAL.parent.mkdir(parents=True, exist_ok=True)
    ds.save_to_disk(str(HF_LOCAL))

# Decode bytes explicitly with soundfile; no automatic audio decoder required.
ds = ds.cast_column("audio", Audio(decode=False))
print(f"Train samples: {len(ds['train'])}")
print(f"Test samples:  {len(ds['test'])}")
print(f"Columns:       {ds['train'].column_names}")


In [ ]:
# Helper: decode a single sample's audio to (array, sampling_rate)
def decode_audio(sample):
    """Decode the raw bytes in a sample's audio field.

    Returns
    -------
    arr : np.ndarray of shape (n_samples,) or (n_samples, n_channels)
    sr  : int sampling rate in Hz
    """
    return sf.read(io.BytesIO(sample["audio"]["bytes"]))


# Helper: read only the header (fast — no audio decode)
def audio_info(sample):
    """Return (sampling_rate, n_frames, duration_s) without decoding the audio."""
    info = sf.info(io.BytesIO(sample["audio"]["bytes"]))
    return info.samplerate, info.frames, info.duration

## 2. Inspect Dataset Structure

Look at the first sample's fields and decode its audio.

In [ ]:
sample = ds['train'][0]

print("Top-level keys:", list(sample.keys()))
print("Audio keys:    ", list(sample['audio'].keys()))
print()

# Non-audio fields
for key, val in sample.items():
    if key != 'audio':
        print(f"  {key}: {val}")

# Audio (decoded)
arr, sr = decode_audio(sample)
print(f"\n  audio:")
print(f"    path:          {sample['audio'].get('path', 'N/A')}")
print(f"    sampling_rate: {sr}")
print(f"    array shape:   {arr.shape}")
print(f"    duration:      {len(arr) / sr:.2f} seconds")

## 3. Species Overview

Count clips per species across train and test splits. This determines which species are viable for each project level.

In [ ]:
# We know the columns from the inspection above
SPECIES_COL = 'species'
LABEL_COL   = 'label'

train_species = ds['train'][SPECIES_COL]   # column access — no per-row Python loop
test_species  = ds['test'][SPECIES_COL]

all_species = list(train_species) + list(test_species)
species_counts = Counter(all_species)
train_counts   = Counter(train_species)
test_counts    = Counter(test_species)

species_df = pd.DataFrame([
    {'species': sp,
     'total':   c,
     'train':   train_counts[sp],
     'test':    test_counts[sp]}
    for sp, c in species_counts.most_common()
])

print(f"Total unique species: {len(species_df)}")
print(f"Total clips:          {len(all_species)}")
print(f"\nClips per species (sorted by count):\n")
print(species_df.to_string(index=False))

In [ ]:
# Visualize class distribution
fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(species_df['species'], species_df['total'], color='steelblue')
ax.set_xlabel('Number of Clips')
ax.set_title('WMMD: Clips per Species')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'species_distribution.png'), dpi=150)
plt.show()

## 4. Taxonomic Grouping

For Level 1 we need each species mapped to a broad biological group. The grouping isn't a pure substring match — **Killer Whale, Pilot Whale, False Killer Whale, Pygmy Killer Whale, Melon-Headed Whale, and Risso's Dolphin** all belong to family **Delphinidae** despite some having "whale" in their common name. The function below encodes those exceptions explicitly. Print any species it can't classify and override manually if needed.

Groups used:
- `baleen` — Mysticeti (Humpback, Blue, Fin, Minke, Bowhead, Right, Gray, Sei)
- `dolphin` — Delphinidae (Bottlenose, Spinner, Striped, Common, Spotted, Killer Whale, Pilot Whale, etc.)
- `toothed_other` — non-Delphinidae Odontoceti (Sperm Whale, Beluga, Narwhal, Beaked Whales)
- `pinniped` — seals, sea lions, walruses

In [ ]:
# Family-level keyword rules. Order matters: Delphinidae specials are checked
# BEFORE the generic 'whale' rule so 'Killer Whale' isn't misclassified as baleen.

DELPHINIDAE_SPECIALS = [
    'killer whale', 'pilot whale', 'false killer', 'pygmy killer',
    'melon-headed', 'melon headed', 'risso', 'rough-toothed', 'rough toothed',
]

OTHER_TOOTHED = [
    'sperm whale', 'beluga', 'narwhal',
    'beaked whale', 'bottlenose whale',   # Ziphiidae — toothed but not Delphinidae
]

BALEEN = [
    'humpback', 'blue whale', 'fin whale', 'finback', 'minke', 'bowhead',
    'right whale', 'gray whale', 'grey whale', 'sei whale',
]

PINNIPED = [
    'seal', 'walrus', 'sea lion',
]


def classify_species(name: str) -> str:
    n = name.lower()
    for kw in DELPHINIDAE_SPECIALS:
        if kw in n:
            return 'dolphin'
    if 'dolphin' in n:
        return 'dolphin'
    for kw in OTHER_TOOTHED:
        if kw in n:
            return 'toothed_other'
    for kw in BALEEN:
        if kw in n:
            return 'baleen'
    for kw in PINNIPED:
        if kw in n:
            return 'pinniped'
    if 'whale' in n:
        return 'unknown_whale'   # flag for manual review
    return 'unknown'


species_df['group'] = species_df['species'].map(classify_species)

# Show any unclassified species — should be empty after manual review
unclassified = species_df[species_df['group'].str.startswith('unknown')]
if len(unclassified) > 0:
    print("!! Unclassified species (review and add to TAXONOMY_OVERRIDES below):")
    print(unclassified.to_string(index=False))
else:
    print("All species classified.")

print()
print(species_df.groupby('group')['total'].agg(['sum', 'count']).rename(
    columns={'sum': 'clips', 'count': 'species'}))

In [ ]:
# Manual overrides — use this if `classify_species` got something wrong
# (e.g., a species name not covered by the keyword rules).
TAXONOMY_OVERRIDES = {
    # 'Exact Species Name From Dataset': 'group_label',
}

for sp, group in TAXONOMY_OVERRIDES.items():
    species_df.loc[species_df['species'] == sp, 'group'] = group

# Final per-group summary
group_summary = (species_df
                 .groupby('group')
                 .agg(species_count=('species', 'count'),
                      total_clips=('total', 'sum'),
                      train_clips=('train', 'sum'),
                      test_clips=('test', 'sum'))
                 .sort_values('total_clips', ascending=False))
print(group_summary)

## 5. Level 2: Delphinidae Subset

Filter to dolphin species only. Identify those with enough clips to be usable for graph SSL — minimum is loosely 10 clips, but the spectral methods will be more reliable on species with ≥20 (especially after segmenting into 2-second windows).

In [ ]:
delphinidae_df = (species_df[species_df['group'] == 'dolphin']
                  .sort_values('total', ascending=False)
                  .reset_index(drop=True))

print(f"Delphinidae species:           {len(delphinidae_df)}")
print(f"Total Delphinidae clips:       {delphinidae_df['total'].sum()}")
print(f"Species with ≥ 10 clips:       {(delphinidae_df['total'] >= 10).sum()}")
print(f"Species with ≥ 20 clips:       {(delphinidae_df['total'] >= 20).sum()}\n")
print(delphinidae_df.to_string(index=False))

## 6. Level 3: Bottlenose Dolphin Clips

Check Bottlenose Dolphin clip count. After 2-second segmentation we want ≥30 windows for unsupervised vocalization-type clustering to be meaningful. If too few, fall back to Spinner Dolphin (also well-represented in WMMD with all three vocalization types) or Sperm Whale (clicks / codas / slow clicks form a different but interesting three-way split).

In [ ]:
# Match by substring — handles 'Bottlenose Dolphin', 'Common Bottlenose Dolphin', etc.
def find_species(df, substring):
    mask = df['species'].str.lower().str.contains(substring.lower())
    return df[mask]


for name in ['bottlenose', 'spinner', 'sperm whale']:
    matches = find_species(species_df, name)
    if len(matches):
        print(f"  {name!r} → {len(matches)} matching species:")
        for _, row in matches.iterrows():
            print(f"      {row['species']}: {row['total']} clips "
                  f"(train={row['train']}, test={row['test']})")
    else:
        print(f"  {name!r} → no match")
    print()

## 7. Audio Properties

Check sampling rates and durations to inform preprocessing decisions (resample target, segmentation window). Uses `sf.info()` which reads only the WAV header — much faster than full decode.

In [ ]:
# Read headers across the full train set (fast — no decoding)
sampling_rates = []
durations = []
n_frames_list = []

for i in range(len(ds['train'])):
    sr, n_frames, dur = audio_info(ds['train'][i])
    sampling_rates.append(sr)
    durations.append(dur)
    n_frames_list.append(n_frames)

sampling_rates = np.array(sampling_rates)
durations      = np.array(durations)

print(f"Sampling rates found: {sorted(set(sampling_rates.tolist()))}")
print(f"\nDuration statistics (seconds):")
print(f"  Min:    {durations.min():.2f}")
print(f"  Max:    {durations.max():.2f}")
print(f"  Mean:   {durations.mean():.2f}")
print(f"  Median: {np.median(durations):.2f}")
print(f"  Std:    {durations.std():.2f}")

# How many clips would survive a 2-second window? How many windows would we get?
n_clips_short = int((durations < 2.0).sum())
n_2s_windows  = int(np.floor(durations / 2.0).sum())
print(f"\nClips shorter than 2s:               {n_clips_short} ({n_clips_short/len(durations)*100:.1f}%)")
print(f"Total non-overlapping 2s windows:    {n_2s_windows}")

In [ ]:
# Distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(durations, bins=40, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Duration (seconds)')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Clip Duration Distribution (n={len(durations)})')
axes[0].axvline(2.0, color='red', linestyle='--', label='2s window')
axes[0].legend()

sr_counts = Counter(sampling_rates.tolist())
axes[1].bar([str(s) for s in sorted(sr_counts)],
            [sr_counts[s] for s in sorted(sr_counts)],
            color='steelblue')
axes[1].set_xlabel('Sampling Rate (Hz)')
axes[1].set_ylabel('Count')
axes[1].set_title('Sampling Rate Distribution')
plt.setp(axes[1].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'audio_properties.png'), dpi=150)
plt.show()

## 8. Listen to Sample Clips

Play one clip from each of the four taxonomic groups for acoustic intuition. (`IPython.display.Audio` is aliased to avoid colliding with the `Audio` schema type imported earlier from `datasets`.)

In [ ]:
from IPython.display import Audio as IPyAudio, display

# Pick the highest-clip species in each group
for group in ['baleen', 'dolphin', 'toothed_other', 'pinniped']:
    group_df = species_df[species_df['group'] == group]
    if len(group_df) == 0:
        continue
    species_name = group_df.iloc[0]['species']

    # Find the first clip of this species in the train split
    idx = next(i for i in range(len(ds['train']))
               if ds['train'][i][SPECIES_COL] == species_name)
    arr, sr = decode_audio(ds['train'][idx])

    # If stereo, average to mono for playback
    if arr.ndim > 1:
        arr = arr.mean(axis=1)

    print(f"[{group}] {species_name}  —  {len(arr)/sr:.2f}s @ {sr} Hz")
    display(IPyAudio(arr, rate=sr))

## 9. Save Corrected Metadata to CSV

The keyword-based `classify_species()` function above misclassifies several species. This cell replaces it with a hardcoded taxonomic mapping (verified against the WMMD species inventory) that adds correct `family` and `genus` columns.

**Columns in the output CSV:**
`split`, `index`, `species`, `genus`, `family`, `label`, `sampling_rate`, `duration_s`, `n_samples`, `path`

In [ ]:
# ============================================================
# CORRECTED METADATA CSV
# ============================================================
# The classify_species() keyword rules misclassified several species
# (e.g., Pilot Whales, Killer Whale, Right Whales ended up as 'unknown_whale').
# This cell builds the metadata CSV from scratch using a hardcoded
# taxonomic mapping verified against the WMMD species inventory.

TAXONOMY = {
    # ── Balaenidae ──────────────────────────────────────────────
    'Bowhead_Whale':                    {'family': 'Balaenidae',      'genus': 'Balaena'},
    'Northern_Right_Whale':             {'family': 'Balaenidae',      'genus': 'Eubalaena'},
    'Southern_Right_Whale':             {'family': 'Balaenidae',      'genus': 'Eubalaena'},
    # ── Balaenopteridae ─────────────────────────────────────────
    'Humpback_Whale':                   {'family': 'Balaenopteridae', 'genus': 'Megaptera'},
    'Fin,_Finback_Whale':               {'family': 'Balaenopteridae', 'genus': 'Balaenoptera'},
    'Minke_Whale':                      {'family': 'Balaenopteridae', 'genus': 'Balaenoptera'},
    # ── Delphinidae ─────────────────────────────────────────────
    'Spinner_Dolphin':                  {'family': 'Delphinidae',     'genus': 'Stenella'},
    'Frasers_Dolphin':                  {'family': 'Delphinidae',     'genus': 'Lagenodelphis'},
    'Striped_Dolphin':                  {'family': 'Delphinidae',     'genus': 'Stenella'},
    'Long-Finned_Pilot_Whale':          {'family': 'Delphinidae',     'genus': 'Globicephala'},
    'Grampus,_Rissos_Dolphin':          {'family': 'Delphinidae',     'genus': 'Grampus'},
    'Short-Finned_Pacific_Pilot_Whale': {'family': 'Delphinidae',     'genus': 'Globicephala'},
    'Pantropical_Spotted_Dolphin':      {'family': 'Delphinidae',     'genus': 'Stenella'},
    'Melon_Headed_Whale':               {'family': 'Delphinidae',     'genus': 'Peponocephala'},
    'Clymene_Dolphin':                  {'family': 'Delphinidae',     'genus': 'Stenella'},
    'False_Killer_Whale':               {'family': 'Delphinidae',     'genus': 'Pseudorca'},
    'Atlantic_Spotted_Dolphin':         {'family': 'Delphinidae',     'genus': 'Stenella'},
    'White-beaked_Dolphin':             {'family': 'Delphinidae',     'genus': 'Lagenorhynchus'},
    'White-sided_Dolphin':              {'family': 'Delphinidae',     'genus': 'Lagenorhynchus'},
    'Common_Dolphin':                   {'family': 'Delphinidae',     'genus': 'Delphinus'},
    'Rough-Toothed_Dolphin':            {'family': 'Delphinidae',     'genus': 'Steno'},
    'Killer_Whale':                     {'family': 'Delphinidae',     'genus': 'Orcinus'},
    'Bottlenose_Dolphin':               {'family': 'Delphinidae',     'genus': 'Tursiops'},
    # ── Monodontidae ────────────────────────────────────────────
    'Beluga,_White_Whale':              {'family': 'Monodontidae',    'genus': 'Delphinapterus'},
    'Narwhal':                          {'family': 'Monodontidae',    'genus': 'Monodon'},
    # ── Odobenidae ──────────────────────────────────────────────
    'Walrus':                           {'family': 'Odobenidae',      'genus': 'Odobenus'},
    # ── Phocidae ────────────────────────────────────────────────
    'Ross_Seal':                        {'family': 'Phocidae',        'genus': 'Ommatophoca'},
    'Harp_Seal':                        {'family': 'Phocidae',        'genus': 'Pagophilus'},
    'Bearded_Seal':                     {'family': 'Phocidae',        'genus': 'Erignathus'},
    'Leopard_Seal':                     {'family': 'Phocidae',        'genus': 'Hydrurga'},
    'Weddell_Seal':                     {'family': 'Phocidae',        'genus': 'Leptonychotes'},
    # ── Physeteridae ────────────────────────────────────────────
    'Sperm_Whale':                      {'family': 'Physeteridae',    'genus': 'Physeter'},
}

# Level 1 group mapping (family → level1 group label)
FAMILY_TO_GROUP = {
    'Balaenidae':      'Balaenidae',
    'Balaenopteridae': 'Balaenopteridae',
    'Delphinidae':     'Delphinidae',
    'Monodontidae':    'Monodontidae',
    'Odobenidae':      'Odobenidae',
    'Phocidae':        'Phocidae',
    'Physeteridae':    'Physeteridae',
}

# ── Build metadata ──────────────────────────────────────────────
rows = []
for split in ['train', 'test']:
    n = len(ds[split])
    for i in range(n):
        sample = ds[split][i]
        species = sample[SPECIES_COL]
        taxon   = TAXONOMY.get(species, {})
        family  = taxon.get('family', 'UNKNOWN')
        genus   = taxon.get('genus',  'UNKNOWN')

        sr, n_frames, dur = audio_info(sample)

        rows.append({
            'split':         split,
            'index':         i,
            'species':       species,
            'genus':         genus,
            'family':        family,
            'label':         sample[LABEL_COL],
            'sampling_rate': sr,
            'duration_s':    dur,
            'n_samples':     n_frames,
            'path':          sample['audio'].get('path'),
        })

metadata = pd.DataFrame(rows)

# ── Sanity checks ───────────────────────────────────────────────
unknown = metadata[metadata['family'] == 'UNKNOWN']
if len(unknown) > 0:
    print(f"WARNING: {len(unknown)} clips have UNKNOWN family:")
    print(unknown['species'].value_counts())
else:
    print("All 1,697 clips mapped to a known family.")

print(f"\nShape: {metadata.shape}")
print(f"Columns: {list(metadata.columns)}")
print(f"\nClips per family:")
print(metadata.groupby('family').size().sort_values(ascending=False).to_string())
print(f"\nClips per species (sorted):")
print(metadata.groupby(['family', 'species']).size().sort_values(ascending=False).to_string())

# ── Save ────────────────────────────────────────────────────────
metadata_path = PROJECT_ROOT / 'wmmd_metadata.csv'
metadata.to_csv(metadata_path, index=False)
print(f"\nSaved corrected metadata to {metadata_path}")


## Next Steps

After running this notebook you should know:

1. **Exact species names** in the dataset → confirm `classify_species` covers all of them; add `TAXONOMY_OVERRIDES` for anything misclassified.
2. **Clip counts per species and per group** → decide minimum-clips threshold for inclusion.
3. **Number of usable Delphinidae species** for Level 2.
4. **Bottlenose Dolphin clip count** for Level 3 (with Spinner / Sperm Whale fallbacks identified).
5. **Sampling rates and durations** → confirm resampling target (22 050 Hz) and segmentation window (2 s).

These decisions feed directly into the preprocessing pipeline (`02_preprocessing.ipynb`).